In [3]:
# ================================================================
# CODE100 UNIT TEST:  L=2 SU(3) Wilson Hessian vs discrete d1^T d1
# Goal:
#   - Build full Wilson Hessian H at the vacuum (theta=0)
#   - Build discrete 1-form operator Delta = d1^T d1 on the same link DOFs
#   - Build the nullspace basis K = (gauge directions) ⊕ (constant-link torons)
#   - Project both operators to the physical complement and verify:
#         H_phys ≈ (1/6) * Delta_phys
#   - Print ranks and spectral checks; assert key invariants.
# ================================================================

import warnings
import numpy as np
# from warnings import ComplexWarning # This import is causing the error

# warnings.filterwarnings("ignore", category=ComplexWarning) # This line depends on the above import

import jax
import jax.numpy as jnp
import jax.scipy as jsp

jax.config.update("jax_enable_x64", True)

# ---------------------------
# SU(3) generators (anti-Hermitian): T_a = (i/2) * lambda_a
# ---------------------------
def su3_generators():
    lam = []
    lam.append(jnp.array([[0, 1, 0],
                          [1, 0, 0],
                          [0, 0, 0]], dtype=jnp.complex128))
    lam.append(jnp.array([[0, -1j, 0],
                          [1j,  0, 0],
                          [0,   0, 0]], dtype=jnp.complex128))
    lam.append(jnp.array([[1,  0, 0],
                          [0, -1, 0],
                          [0,  0, 0]], dtype=jnp.complex128))
    lam.append(jnp.array([[0, 0, 1],
                          [0, 0, 0],
                          [1, 0, 0]], dtype=jnp.complex128))
    lam.append(jnp.array([[0,  0, -1j],
                          [0,  0,  0],
                          [1j, 0,  0]], dtype=jnp.complex128))
    lam.append(jnp.array([[0, 0, 0],
                          [0, 0, 1],
                          [0, 1, 0]], dtype=jnp.complex128))
    lam.append(jnp.array([[0,  0,  0],
                          [0,  0, -1j],
                          [0,  1j, 0]], dtype=jnp.complex128))
    lam.append(jnp.array([[1, 0, 0],
                          [0, 1, 0],
                          [0, 0, -2]], dtype=jnp.complex128) / jnp.sqrt(3.0))
    lam = jnp.stack(lam, axis=0)
    return 1j * lam / 2.0  # anti-Hermitian

T = su3_generators()  # (8,3,3)

def su3_alg_from_vec(a):
    # a: (8,) -> (3,3)
    return jnp.einsum("a,aij->ij", a, T)

def su3_exp(A):
    return jsp.linalg.expm(A)

def build_links(theta_flat, L):
    """
    theta_flat: (L^4 * 4 * 8,) real
    returns U: (L,L,L,L,4,3,3)
    """
    flat = theta_flat.reshape(-1, 8)
    A = jax.vmap(su3_alg_from_vec)(flat)  # (n_links,3,3)
    U = jax.vmap(su3_exp)(A)              # (n_links,3,3)
    return U.reshape(L, L, L, L, 4, 3, 3)

def wilson_action(theta_flat, L, beta=1.0):
    """
    Wilson gauge action:
      S = beta * sum_{plaquettes} [ 1 - (1/3) Re Tr(U_p) ]
    """
    U = build_links(theta_flat, L)
    S = 0.0
    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U[..., mu, :, :]
            U_nu = U[..., nu, :, :]
            U_nu_shift = jnp.roll(U_nu, shift=-1, axis=mu)
            U_mu_shift = jnp.roll(U_mu, shift=-1, axis=nu)

            U_mu_dag_shift = jnp.swapaxes(jnp.conjugate(U_mu_shift), -1, -2)
            U_nu_dag = jnp.swapaxes(jnp.conjugate(U_nu), -1, -2)

            P = U_mu @ U_nu_shift @ U_mu_dag_shift @ U_nu_dag
            trP = jnp.trace(P, axis1=-2, axis2=-1)
            S = S + jnp.sum(1.0 - jnp.real(trP) / 3.0)
    return beta * S

def build_hessian(L, beta=1.0):
    n_params = (L**4) * 4 * 8
    def action_wrap(theta):
        return wilson_action(theta, L, beta=beta)

    grad_S = jax.grad(action_wrap)
    hess_fn = jax.jacfwd(grad_S)

    theta0 = jnp.zeros((n_params,), dtype=jnp.float64)
    H = hess_fn(theta0)
    H = np.array(H, dtype=float)
    H = 0.5 * (H + H.T)
    return H

# ---------------------------
# Gauge directions: alpha(x,a) -> theta(x,mu,a) = alpha(x,a) - alpha(x+mu,a)
# ---------------------------
def alpha_to_theta(alpha, L):
    alpha = np.asarray(alpha, float)  # (L,L,L,L,8)
    theta = np.zeros((L, L, L, L, 4, 8), dtype=float)
    for mu in range(4):
        alpha_fwd = np.roll(alpha, shift=-1, axis=mu)
        theta[..., mu, :] = alpha - alpha_fwd
    return theta.reshape(-1)

def build_gauge_matrix(L):
    n_sites = L**4
    n_color = 8
    n_alpha = n_sites * n_color
    n_theta = n_sites * 4 * n_color
    G = np.zeros((n_theta, n_alpha), dtype=float)

    col = 0
    for x0 in range(L):
        for x1 in range(L):
            for x2 in range(L):
                for x3 in range(L):
                    for a in range(n_color):
                        alpha = np.zeros((L, L, L, L, n_color), dtype=float)
                        alpha[x0, x1, x2, x3, a] = 1.0
                        G[:, col] = alpha_to_theta(alpha, L)
                        col += 1
    return G

def build_constant_link_matrix(L):
    """
    Constant-link (toron) directions: for each (mu,a), theta[...,mu,a] = 1 at all sites.
    """
    n_sites = L**4
    n_color = 8
    n_theta = n_sites * 4 * n_color
    n_const = 4 * n_color
    C = np.zeros((n_theta, n_const), dtype=float)

    col = 0
    for mu in range(4):
        for a in range(n_color):
            theta = np.zeros((L, L, L, L, 4, n_color), dtype=float)
            theta[..., mu, a] = 1.0
            C[:, col] = theta.reshape(-1)
            col += 1
    return C

def matrix_rank(A, tol=1e-10):
    s = np.linalg.svd(A, compute_uv=False)
    return int(np.sum(s > tol))

# ---------------------------
# Discrete d1 operator (curl): 1-forms -> 2-forms on 4D torus
# Ordering for link DOFs:
#    col = (((site*4 + mu)*8) + a)
# ---------------------------
def site_index_4d(x, y, z, t, L):
    return ((x * L + y) * L + z) * L + t

def shift_site(x, y, z, t, mu, L, step):
    coords = [x, y, z, t]
    coords[mu] = (coords[mu] + step) % L
    return coords[0], coords[1], coords[2], coords[3]

def build_d1(L, n_color=8):
    n_sites = L**4
    n_dir = 4
    pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
    n_pairs = len(pairs)

    n_theta = n_sites * n_dir * n_color
    n_plaq = n_sites * n_pairs * n_color
    D = np.zeros((n_plaq, n_theta), dtype=float)

    pair_index = {p: i for i, p in enumerate(pairs)}

    for x in range(L):
        for y in range(L):
            for z in range(L):
                for t in range(L):
                    s = site_index_4d(x, y, z, t, L)
                    for (mu, nu) in pairs:
                        pidx = pair_index[(mu, nu)]

                        x_mu = shift_site(x, y, z, t, mu, L, +1)
                        s_mu_fwd = site_index_4d(*x_mu, L)

                        x_nu = shift_site(x, y, z, t, nu, L, +1)
                        s_nu_fwd = site_index_4d(*x_nu, L)

                        for a in range(n_color):
                            row = ((s * n_pairs + pidx) * n_color + a)

                            col_Anu_fwd = (((s_mu_fwd * 4 + nu) * n_color) + a)
                            col_Anu     = (((s        * 4 + nu) * n_color) + a)
                            col_Amu_fwd = (((s_nu_fwd * 4 + mu) * n_color) + a)
                            col_Amu     = (((s        * 4 + mu) * n_color) + a)

                            # F_{mu nu}(x) = A_nu(x+mu) - A_nu(x) - (A_mu(x+nu) - A_mu(x))
                            D[row, col_Anu_fwd] += 1.0
                            D[row, col_Anu]     -= 1.0
                            D[row, col_Amu_fwd] -= 1.0
                            D[row, col_Amu]     += 1.0
    return D

# ---------------------------
# Main: run the unit test
# ---------------------------
def main():
    np.set_printoptions(precision=6, suppress=True)

    L = 2
    beta = 1.0
    tol_rank = 1e-10
    tol_small = 1e-8

    n_sites = L**4
    n_theta = n_sites * 4 * 8
    print(f"=== CODE100 UNIT TEST ===")
    print(f"L={L}, beta={beta}")
    print(f"n_sites={n_sites}, n_theta={n_theta}")

    print("\n[1] Build Wilson Hessian H at vacuum ...")
    H = build_hessian(L, beta=beta)
    print("H shape:", H.shape)

    print("\n[2] Build discrete Delta = d1^T d1 ...")
    D = build_d1(L, n_color=8)
    Delta = D.T @ D
    Delta = 0.5 * (Delta + Delta.T)
    print("Delta shape:", Delta.shape)
    print("Delta symmetric:", np.allclose(Delta, Delta.T, atol=0, rtol=0))

    print("\n[3] Build gauge matrix G and constant-link matrix C ...")
    G = build_gauge_matrix(L)
    C = build_constant_link_matrix(L)

    # ranks
    rank_G = matrix_rank(G, tol=tol_rank)
    rank_C = matrix_rank(C, tol=tol_rank)
    rank_GC = matrix_rank(np.concatenate([G, C], axis=1), tol=tol_rank)
    dim_intersect = rank_G + rank_C - rank_GC

    print("G shape:", G.shape, "rank(G) =", rank_G, "(expected 120)")
    print("C shape:", C.shape, "rank(C) =", rank_C, "(expected 32)")
    print("rank([G|C]) =", rank_GC, "=> dim(ImG ∩ spanC) =", dim_intersect, "(expected 0)")

    # Check annihilation
    max_Hg = np.linalg.norm(H @ G, axis=0).max()
    max_Dg = np.linalg.norm(Delta @ G, axis=0).max()
    max_Dc = np.linalg.norm(Delta @ C, axis=0).max()
    print(f"max ||H*G||_col = {max_Hg:.3e}")
    print(f"max ||Delta*G||_col = {max_Dg:.3e}")
    print(f"max ||Delta*C||_col = {max_Dc:.3e}")

    print("\n[4] Build physical basis Q_phys = (ImG ⊕ toron)^⊥ ...")
    # Orthonormal basis for Im(G) via SVD
    Ug, sg, _ = np.linalg.svd(G, full_matrices=False)
    Qg = Ug[:, :rank_G]  # (n_theta, rank_G), orthonormal

    # Project constant-link columns orthogonal to gauge
    C_perp = C - Qg @ (Qg.T @ C)
    rank_Cperp = matrix_rank(C_perp, tol=tol_rank)
    print("rank(C_perp) =", rank_Cperp, "(expected 32)")

    K = np.concatenate([Qg, C_perp], axis=1)
    Uk, sk, _ = np.linalg.svd(K, full_matrices=True)
    rank_K = int(np.sum(sk > tol_rank))
    Q_phys = Uk[:, rank_K:]  # orthonormal basis of orthogonal complement

    print("K shape:", K.shape, "rank(K) =", rank_K, "(expected 152)")
    print("Q_phys shape:", Q_phys.shape, "(expected (512, 360))")

    # Assertions for the decomposition dims
    assert rank_G == 120, f"rank(G) unexpected: {rank_G}"
    assert rank_C == 32, f"rank(C) unexpected: {rank_C}"
    assert dim_intersect == 0, f"dim intersection unexpected: {dim_intersect}"
    assert rank_K == 152, f"rank(K) unexpected: {rank_K}"
    assert Q_phys.shape[1] == (n_theta - rank_K), "physical dimension mismatch"

    # Project operators
    print("\n[5] Project to physical sector and compare ...")
    H_phys = Q_phys.T @ H @ Q_phys
    H_phys = 0.5 * (H_phys + H_phys.T)

    Delta_phys = Q_phys.T @ Delta @ Q_phys
    Delta_phys = 0.5 * (Delta_phys + Delta_phys.T)

    wH = np.linalg.eigvalsh(H_phys)
    wD = np.linalg.eigvalsh(Delta_phys)

    print("H_phys:  \u03BB_min =", wH[0], " \u03BB_max =", wH[-1])
    print("D_phys:  \u03BB_min =", wD[0], " \u03BB_max =", wD[-1])

    # Best-fit scalar c in Frobenius inner product: minimize ||H_phys - c Delta_phys||_F
    num = np.sum(H_phys * Delta_phys)
    den = np.sum(Delta_phys * Delta_phys)
    c_est = num / den

    rel_err = np.linalg.norm(H_phys - c_est * Delta_phys, ord="fro") / np.linalg.norm(H_phys, ord="fro")

    print("\nBest-fit proportionality c_est =", c_est, "(expected ~ 1/6 ≈ 0.1666666667)")
    print("Relative Frobenius error ||H_phys - c_est Delta_phys||/||H_phys|| =", rel_err)

    # Eigenvalue-level check (use median ratio for stability)
    ratio_med = np.median(wH / wD)
    ratio_maxdev = np.max(np.abs((wH / wD) - ratio_med))
    print("\nEigen ratio median =", ratio_med, " max deviation =", ratio_maxdev)

    print("\nFirst 10 eigenvalues:")
    print("H_phys:", wH[:10])
    print("D_phys:", wD[:10])
    print("H_phys / D_phys:", (wH[:10] / wD[:10]))

    # Tolerant asserts
    assert abs(c_est - (1.0/6.0)) < 1e-10, f"c_est not ~1/6: {c_est}"
    assert rel_err < 1e-10, f"proportionality error too large: {rel_err}"
    assert max_Hg < 1e-6, f"H does not annihilate gauge directions strongly enough: {max_Hg}"
    assert wH[0] > -1e-10, f"H_phys has significant negative mode: {wH[0]}"
    assert wD[0] > 1e-12, f"Delta_phys not positive definite enough: {wD[0]}"

    print("\nPASS: H_phys ≈ (1/6) * (d1^T d1)_phys and nullspace decomposition matches (gauge ⊕ toron).")

if __name__ == "__main__":
    main()

=== CODE100 UNIT TEST ===
L=2, beta=1.0
n_sites=16, n_theta=512

[1] Build Wilson Hessian H at vacuum ...


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


H shape: (512, 512)

[2] Build discrete Delta = d1^T d1 ...
Delta shape: (512, 512)
Delta symmetric: True

[3] Build gauge matrix G and constant-link matrix C ...
G shape: (512, 128) rank(G) = 120 (expected 120)
C shape: (512, 32) rank(C) = 32 (expected 32)
rank([G|C]) = 152 => dim(ImG ∩ spanC) = 0 (expected 0)
max ||H*G||_col = 3.744e-16
max ||Delta*G||_col = 0.000e+00
max ||Delta*C||_col = 0.000e+00

[4] Build physical basis Q_phys = (ImG ⊕ toron)^⊥ ...
rank(C_perp) = 32 (expected 32)
K shape: (512, 152) rank(K) = 152 (expected 152)
Q_phys shape: (512, 360) (expected (512, 360))

[5] Project to physical sector and compare ...
H_phys:  λ_min = 0.6666666666666605  λ_max = 2.6666666666666727
D_phys:  λ_min = 3.9999999999999805  λ_max = 16.000000000000032

Best-fit proportionality c_est = 0.16666666666666666 (expected ~ 1/6 ≈ 0.1666666667)
Relative Frobenius error ||H_phys - c_est Delta_phys||/||H_phys|| = 5.672722519232499e-16

Eigen ratio median = 0.16666666666666669  max deviation = 8